# Yelp Topic Modeling: BERTopic vs. FASTopic


1. Use a pretrained SentenceTransformer to create review embeddings.
2. Fit **one global BERTopic model** and **one global FASTopic model** on a
   representative California review sample.
3. Compare the models using:
   - C_v coherence
   - NPMI coherence
   - U_Mass coherence
   - Topic diversity
   - Runtime
4. Save both fitted models.
5. Use the selected model to assign topics to all reviews.
6. Export the top topics for every business for use in the Shiny dashboard.

The transformer is used for pretrained embedding inference. BERTopic and
FASTopic still need an unsupervised fit on the Yelp corpus so that the topics
are relevant to Yelp reviews. No hand-labeled topic data is required.

In [ ]:
# Use the PyTorch version already provided by Colab.
# Do not uninstall/reinstall torch or force a CUDA wheel.
!pip -q install -U bertopic fastopic topmost sentence-transformers gensim umap-learn hdbscan pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 129.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 1. Configuration

In [ ]:
from pathlib import Path

DATA_PATH = Path("/content/drive/MyDrive/yelp_reviews_clean_CA.csv")
BUSINESS_PATH = Path(
    "/content/drive/MyDrive/yelp_academic_dataset_business.csv"
)

OUTPUT_DIR = Path("/content/drive/MyDrive/yelp_topic_modeling")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
SEED = 42
N_TOPICS = 25
TOP_N_WORDS = 10

# Run a small test first. Change to False for the final comparison.
TEST_MODE = True

if TEST_MODE:
    MAX_DOCS_FOR_FIT = 5_000
    MAX_REVIEWS_PER_BUSINESS_FOR_FIT = 10
    FASTOPIC_EPOCHS = 10
else:
    MAX_DOCS_FOR_FIT = 50_000
    MAX_REVIEWS_PER_BUSINESS_FOR_FIT = 30
    FASTOPIC_EPOCHS = 50

# Metrics can be expensive on very large corpora.
MAX_DOCS_FOR_COHERENCE = 20_000

# Full-data topic assignment is intentionally disabled during initial testing.
RUN_FULL_DATA_INFERENCE = True

# Select the model after reviewing the comparison table:
# "BERTopic" or "FASTopic"
SELECTED_MODEL_FOR_EXPORT = "BERTopic"

INFERENCE_BATCH_SIZE = 2_000
EMBEDDING_BATCH_SIZE = 128

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Upload yelp_reviews_clean_CA.csv to MyDrive or update DATA_PATH."
    )

print(f"Test mode:       {TEST_MODE}")
print(f"Fit documents:   up to {MAX_DOCS_FOR_FIT:,}")
print(f"Number of topics:{N_TOPICS}")
print(f"Output directory:{OUTPUT_DIR}")

Test mode:       True
Fit documents:   up to 5,000
Number of topics:25
Output directory:/content/drive/MyDrive/yelp_topic_modeling


## 2. Imports, device, and reproducibility

In [ ]:
import gc
import json
import random
import time
import warnings

import numpy as np
import pandas as pd
import polars as pl
import torch

from bertopic import BERTopic
from fastopic import FASTopic
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from topmost.preprocess import Preprocess
from umap import UMAP

warnings.filterwarnings("ignore")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print(f"PyTorch: {torch.__version__}")
print(f"Device:  {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU:     {torch.cuda.get_device_name(0)}")

PyTorch: 2.11.0+cu128
Device:  cuda
GPU:     Tesla T4


## 3. Load and standardize the Yelp data

In [ ]:
reviews_lazy = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=20_000,
    ignore_errors=True,
)

available_columns = reviews_lazy.collect_schema().names()
print("Available columns:")
print(available_columns)


def first_existing(candidates, available, required=False):
    match = next(
        (column for column in candidates if column in available),
        None,
    )
    if required and match is None:
        raise KeyError(
            f"None of the required columns {candidates} were found. "
            f"Available columns: {available}"
        )
    return match


TEXT_COL = first_existing(
    ["text", "review_text", "review"],
    available_columns,
    required=True,
)
BUSINESS_ID_COL = first_existing(
    ["business_id"],
    available_columns,
    required=True,
)
BUSINESS_NAME_COL = first_existing(
    ["business_name", "name"],
    available_columns,
)
DATE_COL = first_existing(
    ["date", "review_date"],
    available_columns,
)

selected_columns = [BUSINESS_ID_COL, TEXT_COL]
if BUSINESS_NAME_COL is not None:
    selected_columns.append(BUSINESS_NAME_COL)
if DATE_COL is not None:
    selected_columns.append(DATE_COL)

reviews_df = (
    reviews_lazy
    .select(selected_columns)
    .filter(pl.col(BUSINESS_ID_COL).is_not_null())
    .filter(pl.col(TEXT_COL).is_not_null())
    .with_columns(
        pl.col(BUSINESS_ID_COL).cast(pl.Utf8),
        pl.col(TEXT_COL).cast(pl.Utf8).str.strip_chars(),
    )
    .filter(pl.col(TEXT_COL).str.len_chars() >= 20)
    .unique(subset=[BUSINESS_ID_COL, TEXT_COL])
    .collect()
)

if BUSINESS_NAME_COL is None:
    if not BUSINESS_PATH.exists():
        raise FileNotFoundError(
            "The review CSV does not contain a business-name column, and "
            f"the business file was not found at {BUSINESS_PATH}."
        )

    business_df = (
        pl.read_csv(
            BUSINESS_PATH,
            columns=["business_id", "name"],
            infer_schema_length=10_000,
            ignore_errors=True,
        )
        .with_columns(
            pl.col("business_id").cast(pl.Utf8),
            pl.col("name").cast(pl.Utf8).str.strip_chars(),
        )
        .unique(subset=["business_id"])
        .rename({"name": "business_name"})
    )

    reviews_df = reviews_df.join(
        business_df,
        left_on=BUSINESS_ID_COL,
        right_on="business_id",
        how="left",
    )
    BUSINESS_NAME_COL = "business_name"

reviews_df = (
    reviews_df
    .with_columns(
        pl.col(BUSINESS_NAME_COL)
        .cast(pl.Utf8)
        .fill_null("Unknown business")
        .str.strip_chars()
    )
)

print(f"Clean review rows:  {reviews_df.height:,}")
print(
    "Unique businesses: "
    f"{reviews_df.get_column(BUSINESS_ID_COL).n_unique():,}"
)
reviews_df.head()

Available columns:
['review_id', 'user_id', 'business_id', 'business_name', 'stars', 'sentiment', 'date', 'text']
Clean review rows:  346,772
Unique businesses: 5,203


business_id,text,business_name,date
str,str,str,str
"""U897DhJaaa3qyr1PVjxfWQ""","""I had the brisket melt for $10…","""Georgia's Smokehouse""","""2015-02-11"""
"""fVUurZgbWXzp2QzDzJzmpA""","""I like their food, but as of l…","""Pickles & Swiss""","""2021-03-05"""
"""SZU9c8V2GuREDN5KgyHFJw""","""Foods are good and nasty, serv…","""Santa Barbara Shellfish Compan…","""2021-11-22"""
"""6JFTijOMHB46yBoyVOjPCA""","""Definitely the most beautiful …","""Santa Barbara County Courthous…","""2014-05-17"""
"""_swrAjNgk60HIGLVzHiqmA""","""My favorite Santa Barbara Thai…","""Khao Kaeng by Empty Bowl Gourm…","""2018-11-08"""


## 4. Create a business-balanced fitting sample

A simple random sample can be dominated by chains and businesses with many
reviews. This sample first caps the reviews contributed by each business and
then applies a global cap.

In [ ]:
shuffled_df = reviews_df.sample(
    fraction=1.0,
    shuffle=True,
    seed=SEED,
)

fit_df = (
    shuffled_df
    .group_by(BUSINESS_ID_COL, maintain_order=True)
    .head(MAX_REVIEWS_PER_BUSINESS_FOR_FIT)
)

if fit_df.height > MAX_DOCS_FOR_FIT:
    fit_df = fit_df.sample(
        n=MAX_DOCS_FOR_FIT,
        shuffle=True,
        seed=SEED,
    )

fit_docs = fit_df.get_column(TEXT_COL).to_list()

print(f"Documents used for fitting: {len(fit_docs):,}")
print(
    "Businesses represented:      "
    f"{fit_df.get_column(BUSINESS_ID_COL).n_unique():,}"
)
print(
    "Average reviews per business:"
    f" {len(fit_docs) / fit_df.get_column(BUSINESS_ID_COL).n_unique():.2f}"
)

Documents used for fitting: 5,000
Businesses represented:      3,270
Average reviews per business: 1.53


## 5. Load the pretrained embedding model

In [ ]:
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=DEVICE,
)

embedding_start = time.perf_counter()

fit_embeddings = embedding_model.encode(
    fit_docs,
    batch_size=EMBEDDING_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

embedding_seconds = time.perf_counter() - embedding_start

print(f"Embedding matrix shape: {fit_embeddings.shape}")
print(f"Embedding time:         {embedding_seconds:.1f} seconds")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Embedding matrix shape: (5000, 384)
Embedding time:         9.3 seconds


## 6. Shared topic-evaluation functions

In [ ]:
def normalize_topic_words(raw_topics, top_n=TOP_N_WORDS):
    """Convert BERTopic/FASTopic topic outputs to list[list[str]]."""

    normalized = []

    if isinstance(raw_topics, dict):
        iterable = [
            value
            for topic_id, value in raw_topics.items()
            if int(topic_id) != -1
        ]
    else:
        iterable = raw_topics

    for topic in iterable:
        if topic is None:
            continue

        if isinstance(topic, str):
            words = topic.split()
        else:
            words = []
            for item in topic:
                if isinstance(item, (tuple, list)) and item:
                    word = item[0]
                else:
                    word = item

                word = str(word).strip()
                if word:
                    words.append(word)

        words = list(dict.fromkeys(words))[:top_n]
        if words:
            normalized.append(words)

    return normalized


def topic_diversity(topic_words, top_n=TOP_N_WORDS):
    truncated = [topic[:top_n] for topic in topic_words if topic]
    total_words = sum(len(topic) for topic in truncated)

    if total_words == 0:
        return np.nan

    unique_words = {
        word
        for topic in truncated
        for word in topic
    }

    return len(unique_words) / total_words


def compute_topic_metrics(
    topic_words,
    documents,
    top_n=TOP_N_WORDS,
):
    """Compute C_v, NPMI, U_Mass, and topic diversity."""

    if len(documents) > MAX_DOCS_FOR_COHERENCE:
        rng = np.random.default_rng(SEED)
        selected_indices = rng.choice(
            len(documents),
            size=MAX_DOCS_FOR_COHERENCE,
            replace=False,
        )
        evaluation_docs = [
            documents[index]
            for index in selected_indices
        ]
    else:
        evaluation_docs = documents

    analyzer = CountVectorizer(
        stop_words="english",
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
    ).build_analyzer()

    tokenized_docs = [
        analyzer(document)
        for document in evaluation_docs
    ]
    tokenized_docs = [
        tokens
        for tokens in tokenized_docs
        if tokens
    ]

    dictionary = Dictionary(tokenized_docs)
    corpus = [
        dictionary.doc2bow(tokens)
        for tokens in tokenized_docs
    ]

    valid_topics = []
    for topic in topic_words:
        valid_words = [
            word
            for word in topic[:top_n]
            if word in dictionary.token2id
        ]
        if len(valid_words) >= 2:
            valid_topics.append(valid_words)

    if not valid_topics:
        return {
            "c_v": np.nan,
            "c_npmi": np.nan,
            "u_mass": np.nan,
            "topic_diversity": np.nan,
            "evaluated_topics": 0,
        }

    common_arguments = {
        "topics": valid_topics,
        "dictionary": dictionary,
        "processes": 1,
    }

    c_v = CoherenceModel(
        texts=tokenized_docs,
        coherence="c_v",
        **common_arguments,
    ).get_coherence()

    c_npmi = CoherenceModel(
        texts=tokenized_docs,
        coherence="c_npmi",
        **common_arguments,
    ).get_coherence()

    u_mass = CoherenceModel(
        corpus=corpus,
        coherence="u_mass",
        **common_arguments,
    ).get_coherence()

    return {
        "c_v": float(c_v),
        "c_npmi": float(c_npmi),
        "u_mass": float(u_mass),
        "topic_diversity": float(
            topic_diversity(valid_topics, top_n=top_n)
        ),
        "evaluated_topics": len(valid_topics),
    }


def topic_table(topic_words, model_name):
    rows = []
    for topic_id, words in enumerate(topic_words):
        rows.append(
            {
                "model": model_name,
                "topic_id": topic_id,
                "topic_label": ", ".join(words[:5]),
                "top_words": ", ".join(words),
            }
        )
    return pd.DataFrame(rows)

## 7. Fit BERTopic

In [ ]:
bertopic_vectorizer = CountVectorizer(
    stop_words="english",
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
)

bertopic_umap = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=SEED,
)

minimum_cluster_size = max(
    15,
    min(100, len(fit_docs) // 200),
)

bertopic_hdbscan = HDBSCAN(
    min_cluster_size=minimum_cluster_size,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

bertopic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=bertopic_umap,
    hdbscan_model=bertopic_hdbscan,
    vectorizer_model=bertopic_vectorizer,
    nr_topics=N_TOPICS,
    top_n_words=TOP_N_WORDS,
    calculate_probabilities=False,
    verbose=True,
)

bertopic_start = time.perf_counter()

bertopic_topics, _ = bertopic_model.fit_transform(
    fit_docs,
    fit_embeddings,
)

bertopic_seconds = time.perf_counter() - bertopic_start

bertopic_topic_words = normalize_topic_words(
    bertopic_model.get_topics()
)

bertopic_metrics = compute_topic_metrics(
    bertopic_topic_words,
    fit_docs,
)

bertopic_outlier_rate = float(
    np.mean(np.asarray(bertopic_topics) == -1)
)

print(f"BERTopic runtime:      {bertopic_seconds:.1f} seconds")
print(f"BERTopic topics:       {len(bertopic_topic_words)}")
print(f"BERTopic outlier rate: {bertopic_outlier_rate:.1%}")
print(bertopic_metrics)

topic_table(
    bertopic_topic_words,
    "BERTopic",
).head(20)

2026-07-28 14:30:36,828 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-28 14:31:12,994 - BERTopic - Dimensionality - Completed ✓
2026-07-28 14:31:12,995 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-28 14:31:13,341 - BERTopic - Cluster - Completed ✓
2026-07-28 14:31:13,342 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-07-28 14:31:14,500 - BERTopic - Representation - Completed ✓
2026-07-28 14:31:14,501 - BERTopic - Topic reduction - Reducing number of topics
2026-07-28 14:31:14,518 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-28 14:31:15,660 - BERTopic - Representation - Completed ✓
2026-07-28 14:31:15,664 - BERTopic - Topic reduction - Reduced number of topics from 27 to 25


BERTopic runtime:      39.4 seconds
BERTopic topics:       24
BERTopic outlier rate: 27.7%
{'c_v': 0.63817339948035, 'c_npmi': 0.09885825422138776, 'u_mass': -3.51733277788149, 'topic_diversity': 0.9203539823008849, 'evaluated_topics': 24}


,model,topic_id,topic_label,top_words
0,BERTopic,0,"car, store, customer, said, business","car, store, customer, said, business, job, sho..."
1,BERTopic,1,"dr, doctor, appointment, pain, care","dr, doctor, appointment, pain, care, office, s..."
2,BERTopic,2,"tacos, food, sandwich, chicken, taco","tacos, food, sandwich, chicken, taco, mexican,..."
3,BERTopic,3,"wine, tasting, wines, bar, tour","wine, tasting, wines, bar, tour, beer, drinks,..."
4,BERTopic,4,"room, hotel, stay, night, clean","room, hotel, stay, night, clean, bed, rooms, a..."
5,BERTopic,5,"hair, cut, salon, haircut, stylist","hair, cut, salon, haircut, stylist, color, hai..."
6,BERTopic,6,"wedding, event, venue, planning, guests","wedding, event, venue, planning, guests, team,..."
7,BERTopic,7,"dog, dogs, cat, pet, cats","dog, dogs, cat, pet, cats, puppy, care, animal..."
8,BERTopic,8,"dr, dentist, dental, teeth, office","dr, dentist, dental, teeth, office, insurance,..."
9,BERTopic,9,"gym, workout, classes, class, workouts","gym, workout, classes, class, workouts, studio..."


## 8. Save BERTopic

In [ ]:
BERTOPIC_MODEL_PATH = OUTPUT_DIR / "bertopic_yelp_model"

bertopic_model.save(
    str(BERTOPIC_MODEL_PATH),
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model=EMBEDDING_MODEL_NAME,
)

topic_table(
    bertopic_topic_words,
    "BERTopic",
).to_csv(
    OUTPUT_DIR / "bertopic_topics.csv",
    index=False,
)

print(f"Saved BERTopic model to: {BERTOPIC_MODEL_PATH}")

Saved BERTopic model to: /content/drive/MyDrive/yelp_topic_modeling/bertopic_yelp_model


## 9. Fit FASTopic

In [ ]:
fastopic_preprocess = Preprocess(
    vocab_size=10_000,
)

fastopic_model = FASTopic(
    num_topics=N_TOPICS,
    preprocess=fastopic_preprocess,
    doc_embed_model=embedding_model,
    device=DEVICE,
    normalize_embeddings=True,
    low_memory=True,
    low_memory_batch_size=2_000,
    verbose=True,
)

fastopic_start = time.perf_counter()

fastopic_raw_topics, fastopic_doc_topic_dist = (
    fastopic_model.fit_transform(
        fit_docs,
        epochs=FASTOPIC_EPOCHS,
        learning_rate=0.01,
    )
)

fastopic_seconds = time.perf_counter() - fastopic_start

fastopic_topic_words = normalize_topic_words(
    fastopic_raw_topics
)

fastopic_metrics = compute_topic_metrics(
    fastopic_topic_words,
    fit_docs,
)

print(f"FASTopic runtime: {fastopic_seconds:.1f} seconds")
print(f"FASTopic topics:  {len(fastopic_topic_words)}")
print(fastopic_metrics)

topic_table(
    fastopic_topic_words,
    "FASTopic",
).head(20)

2026-07-28 14:31:28,203 - FASTopic - use device: cuda
2026-07-28 14:31:28,204 - FASTopic - Using low memory mode.
2026-07-28 14:31:28,205 - FASTopic - First fit the model.
parsing texts: 100%|██████████| 5000/5000 [00:00<00:00, 7582.79it/s]
2026-07-28 14:31:30,428 - TopMost - Real vocab size: 10000
2026-07-28 14:31:30,429 - TopMost - Real training size: 5000 	 avg length: 41.414


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Training FASTopic: 100%|██████████| 10/10 [00:06<00:00,  1.51it/s]


Topic 0: body pain results nails treatment salon cut health nail surgery physical skin gentle neck haircut
Topic 1: cockroach furious ihop chatting stations remake everytime existent yea atrocious nope registers established creepy racist
Topic 2: store shop buy stores sales gift clothes bought sale prices items item sell stock purchase
Topic 3: wedding photos process helped ceremony planning event budget photo events training reception flowers beginning team
Topic 4: branch atm scam banks banking quoted claimed debit verizon funds plumber collections defensive lied idiot
Topic 5: food restaurant ordered breakfast chicken menu beer burger meal sandwich fries eat served sauce meat
Topic 6: study straws lattes mud outdated hostel stench campsite blocks ppl downstairs halls stink entry maneuver
Topic 7: brunch culinary shawarma mahi bagel ceviche divine horchata deliciousness braised sausages glaze brussel vegetarians steaks
Topic 8: car repair cleaning crew tree parts replaced vehicle was

,model,topic_id,topic_label,top_words
0,FASTopic,0,"body, pain, results, nails, treatment","body, pain, results, nails, treatment, salon, ..."
1,FASTopic,1,"cockroach, furious, ihop, chatting, stations","cockroach, furious, ihop, chatting, stations, ..."
2,FASTopic,2,"store, shop, buy, stores, sales","store, shop, buy, stores, sales, gift, clothes..."
3,FASTopic,3,"wedding, photos, process, helped, ceremony","wedding, photos, process, helped, ceremony, pl..."
4,FASTopic,4,"branch, atm, scam, banks, banking","branch, atm, scam, banks, banking, quoted, cla..."
5,FASTopic,5,"food, restaurant, ordered, breakfast, chicken","food, restaurant, ordered, breakfast, chicken,..."
6,FASTopic,6,"study, straws, lattes, mud, outdated","study, straws, lattes, mud, outdated, hostel, ..."
7,FASTopic,7,"brunch, culinary, shawarma, mahi, bagel","brunch, culinary, shawarma, mahi, bagel, cevic..."
8,FASTopic,8,"car, repair, cleaning, crew, tree","car, repair, cleaning, crew, tree, parts, repl..."
9,FASTopic,9,"cameron, valencia, raisin, eos, turtles","cameron, valencia, raisin, eos, turtles, ameri..."


## 10. Save FASTopic

In [ ]:
FAST_TOPIC_MODEL_PATH = OUTPUT_DIR / "fastopic_yelp_model.zip"

fastopic_model.save(
    str(FAST_TOPIC_MODEL_PATH)
)

topic_table(
    fastopic_topic_words,
    "FASTopic",
).to_csv(
    OUTPUT_DIR / "fastopic_topics.csv",
    index=False,
)

print(f"Saved FASTopic model to: {FAST_TOPIC_MODEL_PATH}")

Saved FASTopic model to: /content/drive/MyDrive/yelp_topic_modeling/fastopic_yelp_model.zip


## 11. Compare the models

In [ ]:
comparison_df = pd.DataFrame(
    [
        {
            "model": "BERTopic",
            "fit_documents": len(fit_docs),
            "number_of_topics": len(bertopic_topic_words),
            "c_v": bertopic_metrics["c_v"],
            "c_npmi": bertopic_metrics["c_npmi"],
            "u_mass": bertopic_metrics["u_mass"],
            "topic_diversity": bertopic_metrics[
                "topic_diversity"
            ],
            "fit_seconds": bertopic_seconds,
            "outlier_rate": bertopic_outlier_rate,
        },
        {
            "model": "FASTopic",
            "fit_documents": len(fit_docs),
            "number_of_topics": len(fastopic_topic_words),
            "c_v": fastopic_metrics["c_v"],
            "c_npmi": fastopic_metrics["c_npmi"],
            "u_mass": fastopic_metrics["u_mass"],
            "topic_diversity": fastopic_metrics[
                "topic_diversity"
            ],
            "fit_seconds": fastopic_seconds,
            "outlier_rate": np.nan,
        },
    ]
)

# Higher is better for all four quality metrics, including U_Mass
# where a value closer to zero is generally better.
quality_metrics = [
    "c_v",
    "c_npmi",
    "u_mass",
    "topic_diversity",
]

for metric in quality_metrics:
    comparison_df[f"{metric}_rank"] = (
        comparison_df[metric]
        .rank(ascending=False, method="min")
    )

comparison_df["mean_quality_rank"] = comparison_df[
    [f"{metric}_rank" for metric in quality_metrics]
].mean(axis=1)

comparison_df = comparison_df.sort_values(
    ["mean_quality_rank", "fit_seconds"]
).reset_index(drop=True)

comparison_df.to_csv(
    OUTPUT_DIR / "topic_model_comparison.csv",
    index=False,
)

comparison_df

,model,fit_documents,number_of_topics,c_v,c_npmi,u_mass,topic_diversity,fit_seconds,outlier_rate,c_v_rank,c_npmi_rank,u_mass_rank,topic_diversity_rank,mean_quality_rank
0,BERTopic,5000,24,0.638173,0.098858,-3.517333,0.920354,39.424430,0.2774,1.0,1.0,1.0,2.0,1.25
1,FASTopic,5000,25,0.477216,-0.208796,-13.203868,0.995968,16.028684,NaN,2.0,2.0,2.0,1.0,1.75


### Interpretation

- Higher C_v is better.
- Higher NPMI is better.
- Higher U_Mass is better; it is commonly negative, so values closer to zero
  are better.
- Higher topic diversity is better, but diversity should not be interpreted
  without inspecting coherence.
- Review the topic words and representative documents manually before choosing
  the production model.

## 12. Inspect representative topics

In [ ]:
print("BERTopic topics")
display(topic_table(bertopic_topic_words, "BERTopic").head(15))

print("\nFASTopic topics")
display(topic_table(fastopic_topic_words, "FASTopic").head(15))

print("\nBERTopic representative documents")
for topic_id in range(min(5, len(bertopic_topic_words))):
    representatives = (
        bertopic_model.get_representative_docs(topic_id)
        or []
    )
    print(f"\nTopic {topic_id}: {bertopic_topic_words[topic_id][:5]}")
    for document in representatives[:2]:
        print(" -", document[:300].replace("\n", " "))

BERTopic topics


,model,topic_id,topic_label,top_words
0,BERTopic,0,"car, store, customer, said, business","car, store, customer, said, business, job, sho..."
1,BERTopic,1,"dr, doctor, appointment, pain, care","dr, doctor, appointment, pain, care, office, s..."
2,BERTopic,2,"tacos, food, sandwich, chicken, taco","tacos, food, sandwich, chicken, taco, mexican,..."
3,BERTopic,3,"wine, tasting, wines, bar, tour","wine, tasting, wines, bar, tour, beer, drinks,..."
4,BERTopic,4,"room, hotel, stay, night, clean","room, hotel, stay, night, clean, bed, rooms, a..."
5,BERTopic,5,"hair, cut, salon, haircut, stylist","hair, cut, salon, haircut, stylist, color, hai..."
6,BERTopic,6,"wedding, event, venue, planning, guests","wedding, event, venue, planning, guests, team,..."
7,BERTopic,7,"dog, dogs, cat, pet, cats","dog, dogs, cat, pet, cats, puppy, care, animal..."
8,BERTopic,8,"dr, dentist, dental, teeth, office","dr, dentist, dental, teeth, office, insurance,..."
9,BERTopic,9,"gym, workout, classes, class, workouts","gym, workout, classes, class, workouts, studio..."



FASTopic topics


,model,topic_id,topic_label,top_words
0,FASTopic,0,"body, pain, results, nails, treatment","body, pain, results, nails, treatment, salon, ..."
1,FASTopic,1,"cockroach, furious, ihop, chatting, stations","cockroach, furious, ihop, chatting, stations, ..."
2,FASTopic,2,"store, shop, buy, stores, sales","store, shop, buy, stores, sales, gift, clothes..."
3,FASTopic,3,"wedding, photos, process, helped, ceremony","wedding, photos, process, helped, ceremony, pl..."
4,FASTopic,4,"branch, atm, scam, banks, banking","branch, atm, scam, banks, banking, quoted, cla..."
5,FASTopic,5,"food, restaurant, ordered, breakfast, chicken","food, restaurant, ordered, breakfast, chicken,..."
6,FASTopic,6,"study, straws, lattes, mud, outdated","study, straws, lattes, mud, outdated, hostel, ..."
7,FASTopic,7,"brunch, culinary, shawarma, mahi, bagel","brunch, culinary, shawarma, mahi, bagel, cevic..."
8,FASTopic,8,"car, repair, cleaning, crew, tree","car, repair, cleaning, crew, tree, parts, repl..."
9,FASTopic,9,"cameron, valencia, raisin, eos, turtles","cameron, valencia, raisin, eos, turtles, ameri..."



BERTopic representative documents

Topic 0: ['car', 'store', 'customer', 'said', 'business']
 - I took a small wool shag rug in to be cleaned after about a year of hard use. This place had a few good reviews around the internets, and they had their price list on the web site, which I appreciated. The drop-off went pretty well--I choose my options and pre-paid, noting the passive-aggressive sig
 - I've only written a couple of reviews in my life, but I wanted to take the time to write this one. I'm also a hyper-critical college professor and rarely give A's, never A-pluses. But I have to say, John Biggs and his crew at Anacapa are car restoration WIZARDS; I give them a firm and enthusiastic A

Topic 1: ['dr', 'doctor', 'appointment', 'pain', 'care']
 - This review is specific to Dr. Ehsani who is an extremely unprofessional doctor at JMG. Every test he orders for me I have to call the office multiple times leaving messages for him before he will return my call with the results. Additio

## 13. Assign the selected model to all reviews and export business topics

Leave `RUN_FULL_DATA_INFERENCE = False` during testing. After reviewing the
comparison, set it to `True`, choose `SELECTED_MODEL_FOR_EXPORT`, and rerun this
cell.

This is offline preprocessing. The Shiny app should load the resulting compact
business-level CSV instead of running topic models in real time.

In [ ]:
def infer_bertopic_in_batches(documents):
    all_topic_ids = []
    all_topic_scores = []

    for start in range(0, len(documents), INFERENCE_BATCH_SIZE):
        batch_docs = documents[
            start:start + INFERENCE_BATCH_SIZE
        ]

        batch_embeddings = embedding_model.encode(
            batch_docs,
            batch_size=EMBEDDING_BATCH_SIZE,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )

        batch_topics, batch_probabilities = (
            bertopic_model.transform(
                batch_docs,
                batch_embeddings,
            )
        )

        all_topic_ids.extend(
            int(topic_id)
            for topic_id in batch_topics
        )

        if batch_probabilities is None:
            all_topic_scores.extend(
                [np.nan] * len(batch_topics)
            )
        else:
            probability_array = np.asarray(
                batch_probabilities
            )
            if probability_array.ndim == 1:
                all_topic_scores.extend(
                    probability_array.astype(float).tolist()
                )
            else:
                all_topic_scores.extend(
                    probability_array.max(axis=1)
                    .astype(float)
                    .tolist()
                )

        print(
            f"BERTopic inference: "
            f"{min(start + len(batch_docs), len(documents)):,}/"
            f"{len(documents):,}"
        )

    return all_topic_ids, all_topic_scores


def infer_fastopic_in_batches(documents):
    all_topic_ids = []
    all_topic_scores = []

    for start in range(0, len(documents), INFERENCE_BATCH_SIZE):
        batch_docs = documents[
            start:start + INFERENCE_BATCH_SIZE
        ]

        batch_distribution = np.asarray(
            fastopic_model.transform(batch_docs)
        )

        all_topic_ids.extend(
            batch_distribution.argmax(axis=1)
            .astype(int)
            .tolist()
        )
        all_topic_scores.extend(
            batch_distribution.max(axis=1)
            .astype(float)
            .tolist()
        )

        print(
            f"FASTopic inference: "
            f"{min(start + len(batch_docs), len(documents)):,}/"
            f"{len(documents):,}"
        )

    return all_topic_ids, all_topic_scores


if RUN_FULL_DATA_INFERENCE:
    all_docs = reviews_df.get_column(TEXT_COL).to_list()

    if SELECTED_MODEL_FOR_EXPORT == "BERTopic":
        selected_topic_words = bertopic_topic_words
        topic_ids, topic_scores = infer_bertopic_in_batches(
            all_docs
        )
    elif SELECTED_MODEL_FOR_EXPORT == "FASTopic":
        selected_topic_words = fastopic_topic_words
        topic_ids, topic_scores = infer_fastopic_in_batches(
            all_docs
        )
    else:
        raise ValueError(
            "SELECTED_MODEL_FOR_EXPORT must be "
            "'BERTopic' or 'FASTopic'."
        )

    assigned_df = pd.DataFrame(
        {
            "business_id": reviews_df
            .get_column(BUSINESS_ID_COL)
            .to_list(),
            "business_name": reviews_df
            .get_column(BUSINESS_NAME_COL)
            .to_list(),
            "topic_id": topic_ids,
            "topic_score": topic_scores,
        }
    )

    # BERTopic topic -1 is the outlier topic and is excluded from
    # business-level ranked topics.
    assigned_non_outliers = assigned_df[
        assigned_df["topic_id"] >= 0
    ].copy()

    topic_labels = {
        topic_id: ", ".join(words[:5])
        for topic_id, words in enumerate(
            selected_topic_words
        )
    }

    assigned_non_outliers["topic_label"] = (
        assigned_non_outliers["topic_id"]
        .map(topic_labels)
        .fillna("Unlabeled topic")
    )

    business_topic_summary = (
        assigned_non_outliers
        .groupby(
            [
                "business_id",
                "business_name",
                "topic_id",
                "topic_label",
            ],
            as_index=False,
        )
        .agg(
            topic_review_count=("topic_id", "size"),
            average_topic_score=("topic_score", "mean"),
        )
    )

    business_topic_summary["business_assigned_reviews"] = (
        business_topic_summary
        .groupby("business_id")["topic_review_count"]
        .transform("sum")
    )

    business_topic_summary["topic_share"] = (
        business_topic_summary["topic_review_count"]
        / business_topic_summary["business_assigned_reviews"]
    )

    business_topic_summary["topic_rank"] = (
        business_topic_summary
        .groupby("business_id")["topic_review_count"]
        .rank(
            method="first",
            ascending=False,
        )
        .astype(int)
    )

    business_top_topics = (
        business_topic_summary[
            business_topic_summary["topic_rank"] <= 5
        ]
        .sort_values(
            ["business_name", "topic_rank"]
        )
        .reset_index(drop=True)
    )

    business_top_topics["model"] = (
        SELECTED_MODEL_FOR_EXPORT
    )

    review_output_path = (
        OUTPUT_DIR
        / "review_topic_assignments.parquet"
    )
    business_output_path = (
        OUTPUT_DIR
        / "business_top_topics.csv"
    )

    assigned_df.to_parquet(
        review_output_path,
        index=False,
    )
    business_top_topics.to_csv(
        business_output_path,
        index=False,
    )

    print(f"Saved review assignments: {review_output_path}")
    print(f"Saved business topics:     {business_output_path}")
    display(business_top_topics.head(20))
else:
    print(
        "Full-data inference is disabled. "
        "Set RUN_FULL_DATA_INFERENCE = True after selecting a model."
    )

2026-07-28 14:33:54,148 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-07-28 14:34:15,619 - BERTopic - Dimensionality - Completed ✓
2026-07-28 14:34:15,621 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-07-28 14:34:15,702 - BERTopic - Cluster - Completed ✓


BERTopic inference: 2,000/346,772


2026-07-28 14:34:18,544 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-07-28 14:34:19,476 - BERTopic - Dimensionality - Completed ✓
2026-07-28 14:34:19,478 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-07-28 14:34:19,561 - BERTopic - Cluster - Completed ✓


BERTopic inference: 4,000/346,772


2026-07-28 14:34:23,002 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-07-28 14:34:24,349 - BERTopic - Dimensionality - Completed ✓
2026-07-28 14:34:24,350 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-07-28 14:34:24,473 - BERTopic - Cluster - Completed ✓


...

2026-07-28 14:46:48,914 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-07-28 14:46:49,819 - BERTopic - Dimensionality - Completed ✓
2026-07-28 14:46:49,820 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-07-28 14:46:49,946 - BERTopic - Cluster - Completed ✓


BERTopic inference: 346,000/346,772


2026-07-28 14:46:51,531 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-07-28 14:46:52,004 - BERTopic - Dimensionality - Completed ✓
2026-07-28 14:46:52,006 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-07-28 14:46:52,049 - BERTopic - Cluster - Completed ✓


BERTopic inference: 346,772/346,772
Saved review assignments: /content/drive/MyDrive/yelp_topic_modeling/review_topic_assignments.parquet
Saved business topics:     /content/drive/MyDrive/yelp_topic_modeling/business_top_topics.csv


,business_id,business_name,topic_id,topic_label,topic_review_count,average_topic_score,business_assigned_reviews,topic_share,topic_rank,model
0,tZnPqmwGcLjYeBdwyvhazg,'O' Bar + Kitchen,3,"wine, tasting, wines, bar, tour",8,0.933225,11,0.727273,1,BERTopic
1,tZnPqmwGcLjYeBdwyvhazg,'O' Bar + Kitchen,2,"tacos, food, sandwich, chicken, taco",3,0.705908,11,0.272727,2,BERTopic
2,BZAVgg5ybeX_a4w-FW0mbA,1-800-GOT-JUNK? Central Coast,0,"car, store, customer, said, business",18,0.863037,18,1.000000,1,BERTopic
3,JYhsqscSETHckYeqjGZ5Yg,1-800-GOT-JUNK? Santa Barbara,0,"car, store, customer, said, business",14,0.915802,14,1.000000,1,BERTopic
4,LsniEK_m_Jbj7JMbT4eRxQ,101 Deli,2,"tacos, food, sandwich, chicken, taco",5,0.848800,6,0.833333,1,BERTopic
5,LsniEK_m_Jbj7JMbT4eRxQ,101 Deli,13,"thai, food, chinese, noodles, chicken",1,0.655181,6,0.166667,2,BERTopic
6,hwdWsUws6m2vrvfFz5TjQg,101 Property Mgmt,0,"car, store, customer, said, business",8,0.825245,8,1.000000,1,BERTopic
7,2l-spLLGUzRiZg8dF623Nw,1114 Sports Bar & Games,2,"tacos, food, sandwich, chicken, taco",7,0.798412,22,0.318182,1,BERTopic
8,2l-spLLGUzRiZg8dF623Nw,1114 Sports Bar & Games,3,"wine, tasting, wines, bar, tour",6,0.822234,22,0.272727,2,BERTopic
9,2l-spLLGUzRiZg8dF623Nw,1114 Sports Bar & Games,16,"pizza, crust, pizzas, pizza good, ordered",5,0.603039,22,0.227273,3,BERTopic
